# Embed `cs280-synthetic-circles-v3` with V-JEPA 2.1

**Run on Kaggle GPU (T4).** Requires:
- Internet ON (Kaggle sidebar → Settings)
- Kaggle Secret `HF_TOKEN` (weights + push to HF)

Pipeline:
1. Load `Bmingg/cs280-synthetic-circles-v3` (2000 384×384 images, noisy background) from HuggingFace
2. Load V-JEPA 2.1 ViT-G/16 encoder (frozen) — weights from local `vjepa2_1_vitg_384.pt`
3. Encode every image → `[1408, 24, 24]` patch embeddings  
   *(each image tiled to T=64 frames; 32 temporal tokens averaged → 576 spatial tokens)*
4. Push result to `Bmingg/cs280-synthetic-circles-v3-embeddings-vjepa21-vitg-384` on HuggingFace

Output HF dataset columns: `embedding` (float16, shape 1408×24×24), `cx`, `cy`, `radius`, `color`

## 0. Install V-JEPA 2 from source

Required once per session — `decord` is Linux-only so this must run on Kaggle.

In [ ]:
import subprocess, sys

r = subprocess.run(
    'git clone --depth 1 https://github.com/facebookresearch/vjepa2 /tmp/vjepa2 2>&1 || echo already_cloned',
    shell=True, capture_output=True, text=True)
print(r.stdout)

r = subprocess.run(f'{sys.executable} -m pip install -e /tmp/vjepa2 -q',
                   shell=True, capture_output=True, text=True)
print(r.stdout[-300:] if r.stdout else 'ok')
if r.returncode != 0:
    print('STDERR:', r.stderr[-300:])

## 1. Imports & Constants

In [ ]:
import json
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

SEED = 13
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# V-JEPA 2.1 ViT-G/16 @ 384
IMAGE_SIZE  = 384
PATCH_SIZE  = 16

PATCH_GRID  = IMAGE_SIZE // PATCH_SIZE   # 16
EMBED_DIM   = 1408
PATCH_GRID  = 24           # 384 // 16

# Static-image → video: tile to T=64, tubelet_size=2 → 32 temporal positions
NUM_FRAMES  = 64
N_TEMPORAL  = NUM_FRAMES // 2            # 32

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

EXTRACT_BATCH = 2   # T=64 uses 4× more GPU memory than T=16; keep small

HF_SRC_REPO  = 'Bmingg/cs280-synthetic-circles-v3'
HF_EMB_REPO  = 'Bmingg/cs280-synthetic-circles-v3-embeddings-vjepa21-vitg-384'

COLOR_NAMES = {0: 'red', 1: 'green', 2: 'blue'}

print(f'Device : {DEVICE}')
print(f'CUDA   : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')
print(f'Shapes : image {IMAGE_SIZE}×{IMAGE_SIZE}  patches {PATCH_GRID}×{PATCH_GRID}  embed_dim {EMBED_DIM}')

## 2. Load Source Dataset from HuggingFace

In [ ]:
from datasets import load_dataset

hf_src = load_dataset(HF_SRC_REPO, split='train')
print(hf_src)
print(f'Columns : {hf_src.column_names}')
print(f'Sample 0: cx={hf_src[0]["cx"]}  cy={hf_src[0]["cy"]}  '
      f'radius={hf_src[0]["radius"]}  color={COLOR_NAMES[hf_src[0]["color"]]}')

In [ ]:
class HFCircleDataset(Dataset):
    """Wraps the HF source dataset; returns normalised image tensors + metadata."""
    def __init__(self, hf_dataset, image_size=IMAGE_SIZE):
        self.ds = hf_dataset
        self.H = self.W = image_size

    def __len__(self): return len(self.ds)

    def __getitem__(self, idx):
        row = self.ds[idx]
        img_np = np.array(row['image'].resize((self.H, self.W))).astype(np.float32) / 255.0
        return (torch.from_numpy(img_np.transpose(2, 0, 1)),
                {'cx': row['cx'], 'cy': row['cy'],
                 'radius': row['radius'], 'color': row['color']})


src_ds = HFCircleDataset(hf_src)
img, meta = src_ds[0]
print(f'image tensor: {img.shape}  meta: {meta}')

## 3. Load V-JEPA 2 Encoder

Weights from `facebook/vjepa2-vitl-fpc64-256` on HuggingFace.  
V-JEPA 2 checkpoint key is `target_encoder` (differs from V-JEPA 2.1's `encoder`).

In [ ]:
from huggingface_hub import hf_hub_download, login
from pathlib import Path

# Auth — add HF_TOKEN in Kaggle → Add-ons → Secrets
try:
    from kaggle_secrets import UserSecretsClient
    login(token=UserSecretsClient().get_secret('HF_TOKEN'), add_to_git_credential=False)
    print('HuggingFace login OK')
except Exception as e:
    print(f'HF login skipped ({e})')

# Download V-JEPA 2 weights

VJEPA_WEIGHTS = Path('../vjepa2_1_vitg_384.pt')

result = torch.hub.load('facebookresearch/vjepa2', 'vjepa2_1_vit_giant_384', pretrained=False, trust_repo=True)
encoder = result[0] if isinstance(result, (tuple, list)) else result

ckpt = torch.load(VJEPA_WEIGHTS, map_location='cpu', weights_only=True)
enc_key = 'target_encoder' if 'target_encoder' in ckpt else 'encoder'
sd = {k.replace('module.', '').replace('backbone.', ''): v
      for k, v in ckpt[enc_key].items()}
missing, unexpected = encoder.load_state_dict(sd, strict=False)
assert len(missing) == 0 and len(unexpected) == 0
encoder.eval().to(DEVICE)

for p in encoder.parameters():
    p.requires_grad_(False)

print(f'\nEncoder ready — {sum(p.numel() for p in encoder.parameters())/1e6:.0f}M params on {DEVICE}')

## 4. Encode Images

V-JEPA 2.1 is a video ViT with `tubelet_size=2` trained on T=64 frames.  
Each static image is tiled to T=64; the 32 temporal token positions are averaged  
to recover one `[1408, 24, 24]` spatial feature map per image.

In [ ]:
def encode_images(images_t: torch.Tensor) -> torch.Tensor:
    """
    Args:
        images_t : FloatTensor [B, 3, H, W]  in [0, 1]  on CPU
    Returns:
        FloatTensor [B, 1408, 24, 24]  on CPU
    """
    mean = IMAGENET_MEAN.to(images_t.device)
    std  = IMAGENET_STD.to(images_t.device)
    x = (images_t - mean) / std                               # ImageNet normalise
    x = x.unsqueeze(2).expand(-1, -1, NUM_FRAMES, -1, -1)    # [B,3,64,H,W]
    x = x.contiguous().to(DEVICE)

    with torch.no_grad():
        out = encoder(x)

    tokens = out if isinstance(out, torch.Tensor) else out.last_hidden_state
    tokens = tokens.cpu()   # [B, N_total, D]

    n_total = N_TEMPORAL * PATCH_GRID * PATCH_GRID
    if tokens.shape[1] == n_total + 1:      # drop CLS if present
        tokens = tokens[:, 1:, :]
    elif tokens.shape[1] != n_total:
        raise ValueError(f'Unexpected token count {tokens.shape[1]} (expected {n_total})')

    B, _, D = tokens.shape
    tokens = tokens.view(B, N_TEMPORAL, PATCH_GRID * PATCH_GRID, D)
    tokens = tokens.mean(dim=1)   # average temporal positions → [B, N_spatial, D]
    return tokens.permute(0, 2, 1).reshape(B, D, PATCH_GRID, PATCH_GRID)


# Shape smoke-test
test_imgs = torch.stack([src_ds[i][0] for i in range(2)])
test_emb  = encode_images(test_imgs)
print(f'encode_images output: {test_emb.shape}')   # expect [2, 1408, 24, 24]

## 5. Extract All Embeddings

In [ ]:
loader = DataLoader(src_ds, batch_size=EXTRACT_BATCH, shuffle=False,
                    num_workers=2, pin_memory=True)

all_embeddings: list[torch.Tensor] = []
all_cx, all_cy, all_radius, all_color = [], [], [], []

for images, metas in tqdm(loader, desc='Encoding'):
    emb = encode_images(images)   # [B, 1408, 24, 24] float32 on CPU
    all_embeddings.append(emb.half())  # store as fp16 immediately — never accumulate fp32
    all_cx.extend(metas['cx'].tolist())
    all_cy.extend(metas['cy'].tolist())
    all_radius.extend(metas['radius'].tolist())
    all_color.extend(metas['color'].tolist())

embeddings_fp16 = torch.cat(all_embeddings, dim=0)   # [2000, 1024, 24, 24] fp16
del all_embeddings   # free the per-batch list
torch.cuda.empty_cache()

mb = embeddings_fp16.element_size() * embeddings_fp16.nelement() / 1e6
print(f'Extracted {embeddings_fp16.shape}  ({mb:.0f} MB float16)')

In [ ]:
import json
from pathlib import Path

OUT_DIR = Path('/embeddings')
OUT_DIR.mkdir(parents=True, exist_ok=True)

torch.save(embeddings_fp16, OUT_DIR / 'embeddings.pt')

metadata = [
    {'cx': all_cx[i], 'cy': all_cy[i], 'radius': all_radius[i], 'color': all_color[i]}
    for i in range(len(all_cx))
]
with open(OUT_DIR / 'metadata.json', 'w') as f:
    json.dump(metadata, f)

size_mb = (OUT_DIR / 'embeddings.pt').stat().st_size / 1e6
print(f'Saved to {OUT_DIR}')
print(f'  embeddings.pt : {size_mb:.0f} MB')
print(f'  metadata.json : {len(metadata)} records')
print('Download from Kaggle → Output tab → embeddings/')

## 6. Push Embeddings to HuggingFace

Stored as `float16` to halve dataset size (~2.4 GB vs ~4.7 GB).  
The training notebook converts back to float32 when loading.

**Before running:** `huggingface-cli login` once in the terminal.

In [ ]:
from datasets import Dataset, Features, Array3D, Value

# Convert to numpy — this is a view (no copy), same 2.4 GB
emb_np = embeddings_fp16.numpy()   # [2000, 1024, 24, 24] float16

# Free the torch tensor before building the HF dataset to avoid holding two copies
del embeddings_fp16
torch.cuda.empty_cache()

records = {
    'embedding': [emb_np[i] for i in range(len(emb_np))],
    'cx':        all_cx,
    'cy':        all_cy,
    'radius':    all_radius,
    'color':     all_color,
}

# Free the numpy array — HF dataset now owns the data
del emb_np

features = Features({
    'embedding': Array3D(shape=(EMBED_DIM, PATCH_GRID, PATCH_GRID), dtype='float16'),
    'cx':        Value('int32'),
    'cy':        Value('int32'),
    'radius':    Value('int32'),
    'color':     Value('int8'),
})

emb_hf_ds = Dataset.from_dict(records, features=features)
del records   # free the raw dict after HF has ingested it
print(emb_hf_ds)
print(f'\nPushing to {HF_EMB_REPO} ...')
emb_hf_ds.push_to_hub(HF_EMB_REPO, private=False)
print(f'Done — https://huggingface.co/datasets/{HF_EMB_REPO}')

In [ ]:
emb_hf_ds.push_to_hub(HF_EMB_REPO, private=False)
print(f'Done — https://huggingface.co/datasets/{HF_EMB_REPO}')

## 7. Verify Round-Trip

Load back from HF and check shapes.

In [ ]:
verify_ds = load_dataset(HF_EMB_REPO, split='train')
print(verify_ds)
sample = verify_ds[0]
emb_np = np.array(sample['embedding'])   # [1408, 24, 24]
print(f'embedding shape : {emb_np.shape}  dtype: {emb_np.dtype}')
print(f'cx={sample["cx"]}  cy={sample["cy"]}  radius={sample["radius"]}  color={COLOR_NAMES[sample["color"]]}')